# Miles Nordwall, Nathan Nail ML Midterm Project
TODO
- [ ] Run all best HPs for 1k,5k, and 10k iterations
- [ ] Retest the training time by splitting up the pipeline's steps, training ONLY the
      - pipe.steps[]
      - or grab "Processing Model...total=?"
- [ ] Description of implementation
- [ ] List best hyperparameter settings
    - [ ] MNIST?
    - [ ] Fashion 4/12
- [ ] Task3 Tables + Explanation
- [ ] Task4 compare 8 SVCs bootstrapped vs SVC results from Task3

## Decription of implementation



In [4]:
import numpy as np
import pandas as pd
import os
import time
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist, fashion_mnist

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

#ITERATIONS = [1000,,5000,10000]
#Cs = [10, 1, 0.1, 0.01, 0.001, 0.0001]
GAMMAs = ["scale", 0.005, 0.1, 0.01, 0.001, 0.0005]

SEED = 96
PCA_DIMS = [50,100,200]
#KERNALS = ['linear', 'rbf', 'poly']
KERNALS = ['poly']
#ITERATIONS = [1000,5000,10000]
ITERATIONS = [3000]
#Cs = [100, 10.0, 5.0, 1.0, 0.1, 0.01,0.01, 0.001, 0.0001]
Cs = [2,2,4,5,0.5,1]
#GAMMAs = ["scale", "auto", 0.1, 0.01, 0.01, 0.001, 0.001, 0.001, 0.0001]
#GAMMAs = [0.0005,0.005,0.005,0.001]
#DEGREEs = [1,1,2,2,2,3,4,4,5]
DEGREEs = [1,1,2,2,3,3]
FILE_PATHS = {"mnist" :   {"pca50" : './results/mnist/pca50.csv',
                          "pca100"  : './results/mnist/pca100.csv',
                          "pca200"  : './results/mnist/pca200.csv',
                          "lda"  : './results/mnist/lda.csv',
                          "linear"  : './results/mnist/linear.csv',
                          "rbf"  : './results/mnist/rbf.csv',
                          "poly"  : './results/mnist/poly.csv'},
              "fashion" : {"pca50" : './results/fashion/pca50.csv',
                          "pca100"  : './results/fashion/pca100.csv',
                          "pca200"  : './results/fashion/pca200.csv',
                          "lda"  : './results/fashion/lda.csv',
                          "linear"  : './results/fashion/linear.csv',
                          "rbf"  : './results/fashion/rbf.csv',
                          "poly"  : './results/fashion/poly.csv'}
              }


In [5]:
'''#1'''
# MNIST - 70,000 images (60k are for training) of handwritten digits from 0-9.
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
(X_train_f, Y_train_f), (X_test_f, Y_test_f) = fashion_mnist.load_data()

In [6]:
'''#2'''
#flatten all data
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

X_train_f = X_train_f.reshape(X_train_f.shape[0], -1)
X_test_f = X_test_f.reshape(X_test_f.shape[0], -1)

In [7]:
standard_pl : Pipeline
fashion_pl : Pipeline
sc = StandardScaler()

In [8]:
def save_n_print_results(results, file_path):
    if isinstance(results, dict):
        results = [results]  
    df = pd.concat([pd.DataFrame(r) for r in results if r is not None], ignore_index=True)
    df.to_csv(file_path, mode='a', header=not os.path.exists(file_path), index=False)
    print(df.to_string(index=False))

def plot_t_vs_acc(times, scores, num_iter):
    '''
    Params : times (1D float array) - time cost
             scores (1D float array) - accuracy 
    Makes simple plot.
    Returns : 
    '''
    xs = np.array(times)
    ys = np.array(scores)

    plt.plot(xs, ys)
    return plt
    

def fit_get_time(pipe, x_train, y_train):
    '''
    Params : pipe (Pipeline) - pipeline
             x_train (1D array) - feature training set
             y_train (1D array) - label training set

    Fits pipeline, records the time interval.

    Returns : fit pipeline (1Darray),
              total time (int)
    '''
    start = time.time()
    fit_pipe = pipe.fit(X=x_train, y=y_train)
            
    stop = time.time() 
    time_total = stop - start
    return fit_pipe, time_total

def make_pipe_(sc, dim_red, kern, num_iter, 
               c, gam = 'scale', deg = 3):
    '''
    Params : sc (StandardScaler) - standardization method
             dim_red (PCA | LDA)- dimension reduction method
             kern ('linear' | 'rbf' | 'poly') - kernal
             num_iter (int) - max iterations
             C (float) - SVC regularization hyperparameter 
             Gamma (String | float) - 
    Utility function that makes a pipeline based
    on C, Gamma, and Degree. 

    Returns : Pipeline 

    '''
    pipe = Pipeline([("scaler",sc),
                    ("dim_red",dim_red),
                    ("model", SVC(C=c, gamma=gam, degree=deg, kernel=kern, max_iter=num_iter))],
                    verbose=True)
    return pipe
            

def svc_train(dim_red, kern, sc, data, iterations,fp, shuffle=True):
    '''
    params: dim_red - type of dimension reduction
            kern - kernal 
            sc - StandardScaler
            data - data[0] : X_train
                   data[1] : Y_train
                   data[2] : X_test
                   data[3] : Y_test
            fp - file-path

    Creates the various pipelines, fits, scores, and
    stores all the results in the results_ dictionary 

    Returns : dictionary storing: 
              "dim_red" : dim_red,
              "kernal" : kern,
              "num_iter" : [],
              "cs" : [],
              "gammas" : [],
              "degs" : [],
              "pipes" : [],
              "scores" : [],
              "times" : []
    '''
    if len(Cs) == len(GAMMAs) == len(DEGREEs):
        num_hyperparams = len(Cs)
    else:
        raise ValueError(f"Cs, GAMMAs, DEGREEs must be same length. Got {len(Cs)}, {len(GAMMAs)}, {len(DEGREEs)}")

    #seperate data into train and test sets
    x_train = data[0]
    y_train = data[1]
    x_test = data[2]
    y_test = data[3]


    results_ = {
            "dim_red" : [],
            "kernal" : [],
            "num_iter" : [],
            "cs" : [],
            "gammas" : [],
            "degs" : [],
            "train_scores" : [],
            "test_scores" : [],
            "pipes" : [],
            "times" : []
            }

    print(f"Starting svc_train({str(dim_red)}{kern}{iterations})")
    #shuffle each of the global hyperparameter arrays to save time
    #instead of GridSearch
    #cs      = rand_gen.permutation(Cs)
    #gams    = rand_gen.permutation(GAMMAs)
    #degs    = rand_gen.permutation(DEGREEs)
    if shuffle:
        #no seed, want random to randomly shuffle the arrays 
        print("Permutating hyperparameters...")
        rand_gen = np.random.RandomState()
        idx_c   = rand_gen.permutation(len(Cs))
        idx_g   = rand_gen.permutation(len(GAMMAs))
        idx_d   = rand_gen.permutation(len(DEGREEs))

        cs   = [Cs[i]      for i in idx_c]
        gams = [GAMMAs[i]  for i in idx_g]
        degs = [DEGREEs[i] for i in idx_d]

    
        print(cs)
        print(gams)
        print(degs)
    cs = Cs
    gams = GAMMAs
    degs = DEGREEs

    
    #make pipe -> fit -> get scores -> store results
    for iteration in iterations:   
        for j in range(num_hyperparams):
            c = cs[j]
            gam = gams[j]
            deg = degs[j]
            print("---------------------------------------- Starting... ----------------------------------------")
            print(f"dim_red={str(dim_red)} | num_iter={iteration} | kernal={kern} | C={c} | gamma={gam} | degree={deg}")
            if kern == "linear":
                pipe = make_pipe_(sc=sc, dim_red=dim_red, kern=kern, num_iter=iteration,
                                  c=c)
            elif kern == "rbf":
                pipe = make_pipe_(sc=sc, dim_red=dim_red, kern=kern, num_iter=iteration,
                                  c=c, gam=gam)
            else:   #poly
                pipe = make_pipe_(sc=sc, dim_red=dim_red, kern=kern, num_iter=iteration,
                                  c=c, gam=gam, deg=deg)

            fit_pipe, time_ = fit_get_time(pipe, x_train, y_train)

            train_score =  fit_pipe.score(x_train, y_train)
            test_score =  fit_pipe.score(x_test, y_test)
            

            results_['dim_red'].append(str(dim_red))
            results_['kernal'].append(kern)
            results_['num_iter'].append(iteration)
            results_['cs'].append(c)
            results_['gammas'].append(gam)
            results_['degs'].append(deg)
            results_['pipes'].append(pipe)
            results_['train_scores'].append(train_score)
            results_['test_scores'].append(test_score)
            results_['times'].append(time_)
        
        
            print(f"dim_red={str(dim_red)} | num_iter={iteration} | kernal={kern} | C={c} | gamma={gam} | degree={deg} | time={time_} | train_score= {train_score} | test_score= {test_score}")
            print("---------------------------------------- Complete! ----------------------------------------\n\n")
        
        save_n_print_results(results_, fp)    
    
    return results_


## Best Hyperparamters
Standardization: StandardScaler()

- Made 3 global arrays : Cs, GAMMAs, and DEGREEs
- To get a feel for the hyperparameters, put a wide range of values in each
- Cs[0.0001-100.0] GAMMAs['scale', 'auto', 0.001-10.0] DEGREEs [1-10]
- For the first few rounds of testing, the 3 arrays would be shuffled for the sake of exploration.
- Took insights and 
    - Ran on linear first, found some more useful params
    - Ran on rbf next, tried

- C does
- Gamma does
- Degree does

### MNIST



### Fashion:

| SVC_kernal | Dim_reduction | C    | gamma | degree | train_score | test_score | iterations | time | model_time |
| ---------- | ------------- | ---- | ----- | ------ | ----------- | ---------- | ---------- | ---- | ---------- |
| 'linear'   | PCA50         | 0.01 | __ | __ | NEED | 0.8539 | 5000 | 99.7379 | NEED |
| 'linear'   | PCA100        | 
| 'linear'   | PCA200        |
| 'linear'   | LDA           | 0.01 | __ | __ | NEED | 0.8255 | 5000 |19.5266 | NEED | 
| 'rbf'      | PCA50         | 2.0 | 0.005 | __ | 0.93725 | 0.8802 | 3000 | 54.4010 |  47.1s |
| 'rbf'      | PCA100        | 2.0 | 0.005 | __ | 0.959433 | 0.8854 | 3000 | 112.2138 | 1.7min | 
| 'rbf'      | PCA200        | 2.0 | 0.005 | __ | 0.97615 | 0.8863 | 3000 | 248.997985 | 3.9min | 
| 'rbf'      | LDA           | 1.5 | 0.005 | __ |  0.847  | 0.8286 | 3000 | 29.92428 | 10.7s | 
| 'poly'     | PCA50         |
| 'poly'     | PCA100        |
| 'poly'     | PCA200        |
| 'poly'     | LDA           |

## PCA[50,100,200] vs LDA
- [ ] Table
    - [ ] Time cost of training
    - [ ] Dim Reduction
    - [ ] Test Error
- [ ] Explanation

## SVC Kernels : 'linear' vs 'rbf' vs 'poly'
- [ ] Table
    - [ ] Time cost of training (SVC training time only)
    - [ ] Prediciton error



In [10]:


def run_all_variations(data, folder_name, kerns, pca_comps, num_iter, fpaths):
    '''
    Params: folder_name (String) -  either "fashion" or "mnist"
            data - data[0] : X train
                   data[1] : Y train
                   data[2] : X test
                   data[3] : Y test
            kernals (1D int arr) - list of SVC kernals
            pca_dims (1D int arr) - list of n_components to pass to PCA
            num_iter (1D int arr) - list of max_iterations to run 
            fpaths (1D string arr) - list of file paths 
    '''
    if folder_name != 'fashion' and folder_name != 'mnist':
        raise ValueError(f'Expected either folder_name fashion or mnist but got: {folder_name}')
    
    pca_lda_results = []
    
    for k in kerns:                       # run for all kernal types
        pca_lda_results = []
        for p in pca_comps:               # run for all PCA dimensions 
            pca =  PCA(n_components=p)
            
            if p==50:
                results = svc_train(dim_red=pca,
                                        sc=sc,
                                        kern=k,
                                        data=data,
                                        iterations=num_iter,
                                        fp=fpaths[f'{folder_name}']['pca50'])
            elif p==100: 
                results = svc_train(dim_red=pca,
                                        sc=sc,
                                        kern=k,
                                        data=data,
                                        iterations=num_iter,
                                        fp=fpaths[f'{folder_name}']['pca100'])
            elif p==200: 
                results = svc_train(dim_red=pca,
                                        sc=sc,
                                        kern=k,
                                        data=data,
                                        iterations=num_iter,
                                        fp=fpaths[f'{folder_name}']['pca200'])
        # run for LDA
        lda = LinearDiscriminantAnalysis()
        lda_results = svc_train(dim_red=lda,
                                sc=sc,
                                kern=k,
                                data=data,
                                iterations=num_iter,
                                fp=fpaths[f'{folder_name}']['lda'])
            
        
        pca_lda_results.append(results)
        pca_lda_results.append(lda_results)
        if k=='linear': save_n_print_results(pca_lda_results,   #save linear
                             fpaths[f'{folder_name}']['linear'])
        elif k=='rbf': save_n_print_results(pca_lda_results,    #save rbf
                             fpaths[f'{folder_name}']['rbf'])
        elif k=='poly': save_n_print_results(pca_lda_results,   #save poly
                             fpaths[f'{folder_name}']['poly'])
    
    
    

In [11]:
'''fashion'''
run_all_variations([X_train_f, Y_train_f, X_test_f, Y_test_f], 
                   folder_name='fashion', 
                   kerns=KERNALS, 
                   pca_comps=PCA_DIMS, 
                   num_iter=ITERATIONS, 
                   fpaths=FILE_PATHS)

Starting svc_train(PCA(n_components=50)poly[3000])
Permutating hyperparameters...
[2, 0.5, 5, 4, 2, 1]
[0.001, 'scale', 0.1, 0.005, 0.0005, 0.01]
[1, 2, 2, 3, 1, 3]
---------------------------------------- Starting... ----------------------------------------
dim_red=PCA(n_components=50) | num_iter=3000 | kernal=poly | C=2 | gamma=scale | degree=1
[Pipeline] ............ (step 1 of 3) Processing scaler, total=   0.4s


KeyboardInterrupt: 

In [ ]:
'''mnist'''
run_all_variations([X_train, Y_train, X_test, Y_test], 
                   folder_name='mnist', 
                   kerns=KERNALS, 
                   pca_comps=PCA_DIMS, 
                   num_iter=ITERATIONS, 
                   fpaths=FILE_PATHS)

In [ ]:
'''FASHION ONLY OLD'''
data = [X_train_f, Y_train_f, X_test_f, Y_test_f]
pca_lda_results = []

for k in KERNALS:
    pca_lda_results = []
    for p in PCA_DIMS:
        pca =  PCA(n_components=p)
        
        if p==50:
            results = svc_train(dim_red=pca,
                                    sc=sc,
                                    kern=k,
                                    data=data,
                                    iterations=ITERATIONS,
                                    fp=FILE_PATHS['fashion']['pca50'])
        elif p==100: 
            results = svc_train(dim_red=pca,
                                    sc=sc,
                                    kern=k,
                                    data=data,
                                    iterations=ITERATIONS,
                                    fp=FILE_PATHS['fashion']['pca100'])
        elif p==200: 
            results = svc_train(dim_red=pca,
                                    sc=sc,
                                    kern=k,
                                    data=data,
                                    iterations=ITERATIONS,
                                    fp=FILE_PATHS['fashion']['pca200'])
    lda = LinearDiscriminantAnalysis()
    lda_results = svc_train(dim_red=lda,
                            sc=sc,
                            kern=k,
                            data=data,
                            iterations=ITERATIONS,
                            fp=FILE_PATHS['fashion']['lda'])
        
    
    pca_lda_results.append(results)
    pca_lda_results.append(lda_results)
    if k=='linear': save_n_print_results(pca_lda_results,   #save linear
                         FILE_PATHS['fashion']['linear'])
    elif k=='rbf': save_n_print_results(pca_lda_results,    #save rbf
                         FILE_PATHS['fashion']['rbf'])
    elif k=='poly': save_n_print_results(pca_lda_results,   #save poly
                         FILE_PATHS['fashion']['poly'])
    
    
    

In [ ]:
mnist_all =  [X_train, Y_train, X_test, Y_test]
fashion_all = [X_train_f, Y_train_f, X_test_f, Y_test_f]


In [ ]:
#Task 4 : 8(x3 kernels) SVC Voters vs 1(x3 kernels) SVC
from sklearn.model_selection import KFold

num_sets = 8

# TODO divide MNIST into 8 disjoint sets using Kfold
mnist_X_all = np.concatenate([X_train, X_test], axis=0)
mnist_Y_all = np.concatenate([Y_train, Y_test], axis=0)
kf = KFold(n_splits=num_sets, shuffle=True, random_state=SEED)

mnist_batches = {"x_trains" : [],
                 "y_trains" : [],
                 "x_tests" : [],
                 "y_tests" : []
                }
for train_index, test_index in kf.split(mnist_X_all):
    mnist_batches["x_trains"].append(mnist_X_all[train_index])
    mnist_batches["y_trains"].append(mnist_Y_all[train_index])
    mnist_batches["x_tests"].append(mnist_X_all[test_index])
    mnist_batches["y_tests"].append(mnist_Y_all[test_index])
    


fashion_X_all = np.concatenate([X_train_f, X_test_f], axis=0)
fashion_Y_all = np.concatenate([Y_train_f, Y_test_f], axis=0)
kf_f = KFold(n_splits=num_sets, shuffle=True, random_state=SEED)

fashion_batches = {"x_trains" : [],
                   "y_trains" : [],
                   "x_tests" : [],
                   "y_tests" : []
                  }


for f_train_index, f_test_index in kf_f.split(fashion_X_all):
    fashion_batches["x_trains"].append(fashion_X_all[f_train_index])
    fashion_batches["y_trains"].append(fashion_Y_all[f_train_index])
    fashion_batches["x_tests"].append(fashion_X_all[f_test_index])
    fashion_batches["y_tests"].append(fashion_Y_all[f_test_index])
    
    
# TODO final prediction result obtained by voting



In [ ]:
'''mnist bootstrap'''
i=0
for i in range(num_sets)
    data = [mnist_batches["x_train"][i],
            mnist_batches["y_train"][i],
            mnist_batches["x_test"][i],
            mnist_batches["y_test"][i],
           ]
    run_all_variations(data, 
                       folder_name='mnist', 
                       kerns=KERNALS, 
                       pca_comps=PCA_DIMS, 
                       num_iter=ITERATIONS, 
                       fpaths=FILE_PATHS)

In [ ]:
'''fashion bootstrap'''
i=0
for i in range(num_sets)
    data = [fashion_batches["x_train"][i],
            fashion_batches["y_train"][i],
            fashion_batches["x_test"][i],
            fashion_batches["y_test"][i],
           ]
    run_all_variations(data, 
                       folder_name='fashion', 
                       kerns=KERNALS, 
                       pca_comps=PCA_DIMS, 
                       num_iter=ITERATIONS, 
                       fpaths=FILE_PATHS)